# filename · A — Gemma 4 12B QAT 품질 실험

**설치 → 팀 데이터 업로드 → 설정 변경 → 실행 → 결과 확인** 순서입니다.

평가 사진·프레임·정답은 팀이 준비합니다. 이 노트북은 데이터를 생성하지 않습니다.
처음에는 `MOCK=True`로 파일 읽기와 리포트 생성을 확인하고, 실제 평가는 `False`로 바꾸세요.

선택 모델의 LM Studio 표시명은 `google/gemma-4-12b-qat`입니다. A는 Google의 **QAT 학습을 거친 BF16 체크포인트**를 사용합니다. B에서 실행할 Q4 GGUF와 정밀도가 다르므로 최종 품질은 B에서도 재확인하세요.
[모델과 파일 구분](https://github.com/yooongZa/filename-model-quality-bench/blob/main/docs/MODEL.md)

Based-On: none — 직접 대응하는 LMS 예제가 없는 project_extension입니다.

## 1. 설치

Colab에서 실제 추론할 때는 **런타임 → 런타임 유형 변경 → GPU(A100)** 를 선택하세요.
이 셀은 Colab에서 코드를 받고 필요한 패키지를 설치합니다. 모델 가중치는 실제 실행 단계에서 받습니다.
로컬에서 모의 실행할 때는 저장소 폴더에서 `pip install -r requirements.txt`를 먼저 실행하세요.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False

if IN_COLAB:
    PROJECT = Path("/content/filename-model-quality-bench")
    if not PROJECT.exists():
        subprocess.run([
            "git", "clone", "https://github.com/yooongZa/filename-model-quality-bench.git", str(PROJECT)
        ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements-colab.txt")], check=True)
else:
    # 로컬에서는 현재 폴더 또는 상위 폴더에서 저장소를 찾습니다.
    PROJECT = Path.cwd()
    if not (PROJECT / "a_quality.py").exists():
        PROJECT = PROJECT.parent

assert (PROJECT / "a_quality.py").exists(), "filename-model-quality-bench 폴더에서 실행해 주세요."
sys.path.insert(0, str(PROJECT))
print("코드 위치:", PROJECT)

## 2. 팀 데이터 업로드

아래 폴더를 ZIP으로 묶어 올리세요. 사진만 또는 영상 프레임만 준비해도 됩니다.

```text
evaluation/
  images/
    city.jpg
    picnic.jpg
  frames/
    cycling/
      001.jpg
      002.jpg
      003.jpg
```

- `images/`: 사진 한 장이 평가 한 건입니다.
- `frames/영상이름/`: 같은 영상의 프레임을 시간순 파일명으로 넣습니다. 한 폴더가 평가 한 건입니다.
- MP4를 직접 읽거나 프레임을 자동 생성하지 않습니다. JPG·PNG·WEBP·BMP를 지원합니다.
- 정답은 선택 사항입니다. `answers/샘플ID.json`으로 직접 넣을 수 있습니다.
- Colab에는 사진이 업로드되므로, 팀에서 Colab 사용이 가능한 자료를 선택하세요.

[데이터 형식과 정답 작성법](https://github.com/yooongZa/filename-model-quality-bench/blob/main/docs/DATASET.md)

In [ ]:
import io
import uuid
from filename_bench.uploads import import_dataset_zip
from filename_bench.data import load_cases

# Colab에서 처음 올릴 때는 둘 다 빈 문자열로 둡니다.
LOCAL_ZIP = ""          # 로컬 실행 예: "/내폴더/evaluation.zip"
EXISTING_DATASET = ""   # 이미 가져온 데이터가 있으면 아래 출력된 데이터 경로를 입력합니다.

if EXISTING_DATASET:
    DATASET = Path(EXISTING_DATASET).expanduser().resolve()
else:
    if LOCAL_ZIP:
        archive = Path(LOCAL_ZIP).expanduser()
    elif IN_COLAB:
        from google.colab import files
        uploaded = files.upload()  # 파일 선택 창에서 ZIP 한 개를 선택하세요.
        if len(uploaded) != 1:
            raise ValueError("평가 ZIP 한 개를 선택해 주세요.")
        archive = io.BytesIO(next(iter(uploaded.values())))
    else:
        raise ValueError("LOCAL_ZIP에 준비한 ZIP의 경로를 입력해 주세요.")

    # 다시 업로드해도 기존 데이터에 덮어쓰지 않도록 새 폴더를 만듭니다.
    destination = PROJECT / "data" / "uploads" / uuid.uuid4().hex[:12]
    DATASET = import_dataset_zip(archive, destination)

cases, _ = load_cases(DATASET)
print("데이터 경로:", DATASET)
for case in cases[:20]:
    print(case["id"], "|", case["media_type"], "| 이미지", len(case["paths"]), "장")
print("전체 평가 단위:", len(cases), "건")

## 3. 바꿔 볼 설정

처음에는 **이 셀만 수정**하면 됩니다. 한 번에 한 가지 조건을 바꾸세요.
`SUITE`는 실험 묶음입니다. `smoke`로 확인한 뒤 `resolution`, `vision`, `output`, `frames`, `selection` 중 하나를 실행합니다.

`frames` 실험의 기본 비교값은 1·3·6·12장, `selection`은 후보 중 6장을 선택합니다. 보유한 프레임 수에 맞춰 비교값을 바꾸거나 충분한 후보를 올리세요. 부족한 프레임은 `unsupported`로 기록합니다.

In [ ]:
from datetime import datetime
from filename_bench.common import load_config
from filename_bench.runner import run

MOCK = True                 # True: GPU 없이 흐름 확인 / False: 실제 모델 평가
SUITE = "smoke"             # smoke, baseline, resolution, vision, output, frames, selection
CASE_IDS = None             # None: 업로드 전체 / 일부만 평가하려면 ["city", "cycling"]
SHORT_EDGE_PX = 1080         # 입력 이미지의 짧은 변. 원본보다 확대하지 않습니다.
VISION_TOKEN_BUDGET = 1120  # 이미지 한 장의 시각 토큰 상한: 70, 140, 280, 560, 1120
VIDEO_FRAMES = 3            # 같은 영상에서 한 요청에 넣을 프레임 수
MAX_NEW_TOKENS = 256        # 답변에 허용할 최대 토큰 수. JSON이 잘리면 늘리세요.
CONTEXT_LENGTH = 32768      # 입력과 출력 토큰을 합친 상한
MAX_NEW_REQUESTS = None     # None: 모두 / 2: 이번 실행은 최대 두 요청
RUN_NAME = datetime.now().strftime("%Y%m%d_%H%M%S_%f")  # 설정을 바꾸면 새 결과 폴더 사용

config = load_config(PROJECT / "configs/a_quality.json")
config.update(dataset_root=str(DATASET), run_dir=str(PROJECT / "runs" / RUN_NAME), case_ids=CASE_IDS)
config["baseline"].update(
    short_edge_px=SHORT_EDGE_PX, vision_token_budget=VISION_TOKEN_BUDGET,
    frames=VIDEO_FRAMES, max_new_tokens=MAX_NEW_TOKENS, context_per_request=CONTEXT_LENGTH,
)

# 자동 비교할 후보 목록입니다. 해당 SUITE에서 이 값 하나만 바꿔 반복합니다.
config["sweeps"]["short_edge_px"] = [1080, 720, 480, 360, 240]
config["sweeps"]["vision_token_budget"] = [1120, 560, 280, 140, 70]
config["sweeps"]["max_new_tokens"] = [512, 256, 128]
config["sweeps"]["frames"] = [1, 3, 6, 12]  # 세 프레임만 있으면 [1, 3]으로 수정

print("모델:", config["backend"]["model_label"])
print("실제 로드할 체크포인트:", config["backend"]["model_id"])
print("추론 정밀도: BF16 / QAT 학습 체크포인트")
print("결과 위치:", config["run_dir"])

### 실행 전 확인

모델을 로드하지 않고 입력 개수와 요청 수를 확인합니다. 사진만 올렸다면 영상용 조건은 건너뜁니다.
정답이 없으면 의미 정확도 점수는 비워 두고 `review.csv`에서 사람이 평가합니다.

In [ ]:
plan = run(config, "a", suite=SUITE, mock=MOCK, dry_run=True)

## 4. 실행

`MOCK=False`이면 모델을 다운로드하고 실제 추론합니다. Hugging Face 모델 페이지에서 약관 동의가 필요할 수 있습니다. 접근 오류가 나면 계정 권한을 확인하고 Colab 보안 비밀에 `HF_TOKEN`을 등록하세요. 토큰을 코드에 직접 적지 마세요.

이 실행 셀을 다시 실행하면 같은 설정에서 미완료 요청만 이어갑니다. 위 설정 셀을 다시 실행하면 새 실험이 됩니다.

In [ ]:
result = run(
    config, "a", suite=SUITE, mock=MOCK,
    resume=True, limit=MAX_NEW_REQUESTS,
)
print(result)
if result["stopped_on_error"]:
    print("실행 오류로 중단했습니다. 결과 폴더의 events.jsonl과 report.html을 확인하세요.")

## 5. 결과 확인

- `report.html`: 조건별 성공률·시간·응답 원문
- `summary.csv`: 조건별 요약 / `requests.jsonl`: 요청별 원문
- `review.csv`: 태그 정확도·핵심 누락·근거 없는 추정 등을 사람이 평가
- `config.json`, `backend_*.json`: 설정·모델·실행 환경 기록

모의 실행 화면에는 **모델 성능 결과 아님**이 표시됩니다. 실제 응답의 JSON 성공률과 내용 정확도를 따로 보세요.

In [ ]:
from IPython.display import HTML, display

run_directory = Path(result["run_dir"])
report_path = run_directory / "report.html"
print("리포트:", report_path)
display(HTML(report_path.read_text(encoding="utf-8")))

### 결과를 내려받으려면

아래 셀은 결과 폴더만 ZIP으로 묶습니다. 업로드한 원본 이미지는 포함하지 않습니다. 실행 설정에는 데이터 경로가 기록되므로 외부 공유 전 확인하세요.

In [ ]:
import shutil

result_zip = shutil.make_archive(str(run_directory) + "_report", "zip", run_directory)
if IN_COLAB:
    from google.colab import files
    files.download(result_zip)
else:
    print("결과 ZIP:", result_zip)